In [2]:
import numpy as np
import pywt
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import os

root = r"C:\Users\Admin\Documents\Thesis\LOSO_CV\chb13\fold_test_sid_6"

# =========================================================
# 1) LOAD DATA
# =========================================================
name_data = "train"
# name_data = "validation"
# name_data = "test"

data_path = os.path.join(root, f"{name_data}.npz")

data = np.load(data_path)

if "X" not in data:
    raise KeyError("File npz không chứa key 'X'")

X = data["X"].astype(np.float32)   # shape: (N, 18, 1280)

if X.ndim != 3:
    raise ValueError(f"X phải có shape (N, C, T), nhưng nhận được {X.shape}")

# Label nếu có thì load luôn
y = data["y"].astype(np.int64) if "y" in data else None

N, n_channels, seq_len = X.shape
print("X shape:", X.shape)

if y is not None:
    print("y shape:", y.shape)
    if len(y) != N:
        raise ValueError(f"Số lượng label y ({len(y)}) không khớp số mẫu X ({N})")

# =========================================================
# 2) OUTPUT CONFIG
# =========================================================
root_path = Path(data_path).parent

SAVE_AS_IMAGES = True
SAVE_AS_NPZ = False

# True  -> lưu (N, 1, H, W)
# False -> lưu (N, H, W, 1)
SAVE_CHW = True

img_out_path = root_path / f"{name_data}_wavelet_gray_custom_band"
img_out_path.mkdir(parents=True, exist_ok=True)

npz_out_path = root_path / f"{name_data}_wavelet_gray_custom_band.npz"

# =========================================================
# 3) WAVELET CONFIG
# =========================================================
fs = 128.0
wavelet_name = "cmor1.5-1.0"
tile_size = 224
use_log_power = True

# ----------------------------
# DẢI TẦN TÙY CHỈNH
# ví dụ:
# delta: 0.5-4
# theta: 4-8
# alpha: 8-13
# beta : 13-30
# gamma: 30-45
# ----------------------------
CUSTOM_BAND_NAME = "alpha_beta"
CUSTOM_BAND_LOW = 8
CUSTOM_BAND_HIGH = 30

n_freqs_in_band = 16

add_separator = True
separator_size = 2

RESAMPLE = Image.Resampling.BILINEAR if hasattr(Image, "Resampling") else Image.BILINEAR

# =========================================================
# 4) HELPER
# =========================================================
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    img_min = img.min()
    img_max = img.max()

    if img_max - img_min < 1e-8:
        return np.zeros_like(img, dtype=np.uint8)

    img = (img - img_min) / (img_max - img_min)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    return img


def get_band_freqs(band_low: float, band_high: float, n_freqs: int) -> np.ndarray:
    if n_freqs <= 0:
        raise ValueError("n_freqs phải > 0")
    if band_low >= band_high:
        raise ValueError(f"band_low phải < band_high, nhưng nhận được {band_low} >= {band_high}")

    if n_freqs == 1:
        return np.array([(band_low + band_high) / 2.0], dtype=np.float32)

    return np.linspace(band_low, band_high, n_freqs, dtype=np.float32)


def compute_band_scalogram(
    signal_1d: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs: int,
    use_log_power: bool = True,
) -> np.ndarray:
    freqs_hz = get_band_freqs(band_low, band_high, n_freqs)
    scales = pywt.frequency2scale(wavelet_name, freqs_hz / fs).astype(np.float32)

    coeffs, _ = pywt.cwt(
        data=signal_1d,
        scales=scales,
        wavelet=wavelet_name,
        sampling_period=1.0 / fs,
        method="fft"
    )

    power = np.abs(coeffs) ** 2

    if use_log_power:
        power = np.log1p(power)

    power = power[::-1, :]  # tần số cao ở trên
    return power.astype(np.float32)


def make_gray_band_vertical_scalogram(
    window_norm: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs_in_band: int = 16,
    use_log_power: bool = True,
    add_separator: bool = True,
    separator_size: int = 2,
    tile_size: int = 512,
) -> np.ndarray:
    """
    window_norm: (C, T)

    Output:
      gray_img: (H, W), uint8
    """
    channel_maps = []

    for ch in range(window_norm.shape[0]):
        signal_1d = window_norm[ch]

        power = compute_band_scalogram(
            signal_1d=signal_1d,
            band_low=band_low,
            band_high=band_high,
            fs=fs,
            wavelet_name=wavelet_name,
            n_freqs=n_freqs_in_band,
            use_log_power=use_log_power,
        )

        channel_maps.append(power)

        if add_separator and ch < window_norm.shape[0] - 1:
            sep = np.zeros((separator_size, power.shape[1]), dtype=np.float32)
            channel_maps.append(sep)

    vertical_img = np.concatenate(channel_maps, axis=0)   # (H_raw, W_raw)
    vertical_img_uint8 = normalize_to_uint8(vertical_img)

    img_pil = Image.fromarray(vertical_img_uint8, mode="L")
    img_pil = img_pil.resize((tile_size, tile_size), RESAMPLE)

    gray_img = np.array(img_pil, dtype=np.uint8)   # (H, W)
    return gray_img


# =========================================================
# 5) GENERATE
# =========================================================
wavelet_data_list = []

for win in tqdm(range(N), desc=f"Processing Wavelet Gray ({CUSTOM_BAND_NAME})"):
    window_data = X[win]   # (18, 1280)

    # normalize từng channel EEG
    mean = window_data.mean(axis=1, keepdims=True)
    std = window_data.std(axis=1, keepdims=True)
    window_norm = (window_data - mean) / (std + 1e-8)

    gray_img = make_gray_band_vertical_scalogram(
        window_norm=window_norm,
        band_low=CUSTOM_BAND_LOW,
        band_high=CUSTOM_BAND_HIGH,
        fs=fs,
        wavelet_name=wavelet_name,
        n_freqs_in_band=n_freqs_in_band,
        use_log_power=use_log_power,
        add_separator=add_separator,
        separator_size=separator_size,
        tile_size=tile_size,
    )  # (H, W)

    if SAVE_AS_IMAGES:
        save_path = img_out_path / f"image_{win:05d}.jpg"
        Image.fromarray(gray_img, mode="L").save(save_path, format="JPEG", quality=95)

    if SAVE_AS_NPZ:
        if SAVE_CHW:
            gray_tensor = gray_img[None, :, :]      # (1, H, W)
        else:
            gray_tensor = gray_img[:, :, None]      # (H, W, 1)

        wavelet_data_list.append(gray_tensor)

# =========================================================
# 6) SAVE NPZ
# =========================================================
if SAVE_AS_NPZ:
    X_wavelet = np.stack(wavelet_data_list, axis=0).astype(np.uint8)

    save_dict = {
        "X": X_wavelet
    }

    if y is not None:
        save_dict["y"] = y

    # copy thêm metadata cũ nếu có
    for key in ["sfreq", "window_start_sec", "window_end_sec", "source_edf", "patient_id"]:
        if key in data:
            save_dict[key] = data[key]

    np.savez_compressed(npz_out_path, **save_dict)

    print(f"Saved wavelet npz to: {npz_out_path}")
    print(f"Saved X shape: {X_wavelet.shape}")
    if y is not None:
        print(f"Saved y shape: {y.shape}")

if SAVE_AS_IMAGES:
    print(f"Saved {N} grayscale images to: {img_out_path}")

print("Done.")

X shape: (2772, 18, 2048)
y shape: (2772,)


Processing Wavelet Gray (alpha_beta): 100%|██████████| 2772/2772 [02:29<00:00, 18.58it/s]

Saved 2772 grayscale images to: C:\Users\Admin\Documents\Thesis\LOSO_CV\chb13\fold_test_sid_6\train_wavelet_gray_custom_band
Done.


In [3]:
# =========================================================
# 1) LOAD DATA
# =========================================================
# name_data = "train"
name_data = "validation"
# name_data = "test"

data_path = os.path.join(root, f"{name_data}.npz")

data = np.load(data_path)

if "X" not in data:
    raise KeyError("File npz không chứa key 'X'")

X = data["X"].astype(np.float32)   # shape: (N, 18, 1280)

if X.ndim != 3:
    raise ValueError(f"X phải có shape (N, C, T), nhưng nhận được {X.shape}")

# Label nếu có thì load luôn
y = data["y"].astype(np.int64) if "y" in data else None

N, n_channels, seq_len = X.shape
print("X shape:", X.shape)

if y is not None:
    print("y shape:", y.shape)
    if len(y) != N:
        raise ValueError(f"Số lượng label y ({len(y)}) không khớp số mẫu X ({N})")

# =========================================================
# 2) OUTPUT CONFIG
# =========================================================
root_path = Path(data_path).parent

SAVE_AS_IMAGES = True
SAVE_AS_NPZ = False

# True  -> lưu (N, 1, H, W)
# False -> lưu (N, H, W, 1)
SAVE_CHW = True

img_out_path = root_path / f"{name_data}_wavelet_gray_custom_band"
img_out_path.mkdir(parents=True, exist_ok=True)

npz_out_path = root_path / f"{name_data}_wavelet_gray_custom_band.npz"

# =========================================================
# 3) WAVELET CONFIG
# =========================================================
fs = 128.0
wavelet_name = "cmor1.5-1.0"
tile_size = 224
use_log_power = True

# ----------------------------
# DẢI TẦN TÙY CHỈNH
# ví dụ:
# delta: 0.5-4
# theta: 4-8
# alpha: 8-13
# beta : 13-30
# gamma: 30-45
# ----------------------------
CUSTOM_BAND_NAME = "alpha_beta"
CUSTOM_BAND_LOW = 8
CUSTOM_BAND_HIGH = 30

n_freqs_in_band = 16

add_separator = True
separator_size = 2

RESAMPLE = Image.Resampling.BILINEAR if hasattr(Image, "Resampling") else Image.BILINEAR

# =========================================================
# 4) HELPER
# =========================================================
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    img_min = img.min()
    img_max = img.max()

    if img_max - img_min < 1e-8:
        return np.zeros_like(img, dtype=np.uint8)

    img = (img - img_min) / (img_max - img_min)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    return img


def get_band_freqs(band_low: float, band_high: float, n_freqs: int) -> np.ndarray:
    if n_freqs <= 0:
        raise ValueError("n_freqs phải > 0")
    if band_low >= band_high:
        raise ValueError(f"band_low phải < band_high, nhưng nhận được {band_low} >= {band_high}")

    if n_freqs == 1:
        return np.array([(band_low + band_high) / 2.0], dtype=np.float32)

    return np.linspace(band_low, band_high, n_freqs, dtype=np.float32)


def compute_band_scalogram(
    signal_1d: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs: int,
    use_log_power: bool = True,
) -> np.ndarray:
    freqs_hz = get_band_freqs(band_low, band_high, n_freqs)
    scales = pywt.frequency2scale(wavelet_name, freqs_hz / fs).astype(np.float32)

    coeffs, _ = pywt.cwt(
        data=signal_1d,
        scales=scales,
        wavelet=wavelet_name,
        sampling_period=1.0 / fs,
        method="fft"
    )

    power = np.abs(coeffs) ** 2

    if use_log_power:
        power = np.log1p(power)

    power = power[::-1, :]  # tần số cao ở trên
    return power.astype(np.float32)


def make_gray_band_vertical_scalogram(
    window_norm: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs_in_band: int = 16,
    use_log_power: bool = True,
    add_separator: bool = True,
    separator_size: int = 2,
    tile_size: int = 512,
) -> np.ndarray:
    """
    window_norm: (C, T)

    Output:
      gray_img: (H, W), uint8
    """
    channel_maps = []

    for ch in range(window_norm.shape[0]):
        signal_1d = window_norm[ch]

        power = compute_band_scalogram(
            signal_1d=signal_1d,
            band_low=band_low,
            band_high=band_high,
            fs=fs,
            wavelet_name=wavelet_name,
            n_freqs=n_freqs_in_band,
            use_log_power=use_log_power,
        )

        channel_maps.append(power)

        if add_separator and ch < window_norm.shape[0] - 1:
            sep = np.zeros((separator_size, power.shape[1]), dtype=np.float32)
            channel_maps.append(sep)

    vertical_img = np.concatenate(channel_maps, axis=0)   # (H_raw, W_raw)
    vertical_img_uint8 = normalize_to_uint8(vertical_img)

    img_pil = Image.fromarray(vertical_img_uint8, mode="L")
    img_pil = img_pil.resize((tile_size, tile_size), RESAMPLE)

    gray_img = np.array(img_pil, dtype=np.uint8)   # (H, W)
    return gray_img


# =========================================================
# 5) GENERATE
# =========================================================
wavelet_data_list = []

for win in tqdm(range(N), desc=f"Processing Wavelet Gray ({CUSTOM_BAND_NAME})"):
    window_data = X[win]   # (18, 1280)

    # normalize từng channel EEG
    mean = window_data.mean(axis=1, keepdims=True)
    std = window_data.std(axis=1, keepdims=True)
    window_norm = (window_data - mean) / (std + 1e-8)

    gray_img = make_gray_band_vertical_scalogram(
        window_norm=window_norm,
        band_low=CUSTOM_BAND_LOW,
        band_high=CUSTOM_BAND_HIGH,
        fs=fs,
        wavelet_name=wavelet_name,
        n_freqs_in_band=n_freqs_in_band,
        use_log_power=use_log_power,
        add_separator=add_separator,
        separator_size=separator_size,
        tile_size=tile_size,
    )  # (H, W)

    if SAVE_AS_IMAGES:
        save_path = img_out_path / f"image_{win:05d}.jpg"
        Image.fromarray(gray_img, mode="L").save(save_path, format="JPEG", quality=95)

    if SAVE_AS_NPZ:
        if SAVE_CHW:
            gray_tensor = gray_img[None, :, :]      # (1, H, W)
        else:
            gray_tensor = gray_img[:, :, None]      # (H, W, 1)

        wavelet_data_list.append(gray_tensor)

# =========================================================
# 6) SAVE NPZ
# =========================================================
if SAVE_AS_NPZ:
    X_wavelet = np.stack(wavelet_data_list, axis=0).astype(np.uint8)

    save_dict = {
        "X": X_wavelet
    }

    if y is not None:
        save_dict["y"] = y

    # copy thêm metadata cũ nếu có
    for key in ["sfreq", "window_start_sec", "window_end_sec", "source_edf", "patient_id"]:
        if key in data:
            save_dict[key] = data[key]

    np.savez_compressed(npz_out_path, **save_dict)

    print(f"Saved wavelet npz to: {npz_out_path}")
    print(f"Saved X shape: {X_wavelet.shape}")
    if y is not None:
        print(f"Saved y shape: {y.shape}")

if SAVE_AS_IMAGES:
    print(f"Saved {N} grayscale images to: {img_out_path}")

print("Done.")

X shape: (420, 18, 2048)
y shape: (420,)


Processing Wavelet Gray (alpha_beta): 100%|██████████| 420/420 [00:20<00:00, 20.02it/s]

Saved 420 grayscale images to: C:\Users\Admin\Documents\Thesis\LOSO_CV\chb13\fold_test_sid_6\validation_wavelet_gray_custom_band
Done.


In [4]:
# =========================================================
# 1) LOAD DATA
# =========================================================
# name_data = "train"
# name_data = "validation"
name_data = "test"

data_path = os.path.join(root, f"{name_data}.npz")

data = np.load(data_path)

if "X" not in data:
    raise KeyError("File npz không chứa key 'X'")

X = data["X"].astype(np.float32)   # shape: (N, 18, 1280)

if X.ndim != 3:
    raise ValueError(f"X phải có shape (N, C, T), nhưng nhận được {X.shape}")

# Label nếu có thì load luôn
y = data["y"].astype(np.int64) if "y" in data else None

N, n_channels, seq_len = X.shape
print("X shape:", X.shape)

if y is not None:
    print("y shape:", y.shape)
    if len(y) != N:
        raise ValueError(f"Số lượng label y ({len(y)}) không khớp số mẫu X ({N})")

# =========================================================
# 2) OUTPUT CONFIG
# =========================================================
root_path = Path(data_path).parent

SAVE_AS_IMAGES = True
SAVE_AS_NPZ = False

# True  -> lưu (N, 1, H, W)
# False -> lưu (N, H, W, 1)
SAVE_CHW = True

img_out_path = root_path / f"{name_data}_wavelet_gray_custom_band"
img_out_path.mkdir(parents=True, exist_ok=True)

npz_out_path = root_path / f"{name_data}_wavelet_gray_custom_band.npz"

# =========================================================
# 3) WAVELET CONFIG
# =========================================================
fs = 128.0
wavelet_name = "cmor1.5-1.0"
tile_size = 224
use_log_power = True

# ----------------------------
# DẢI TẦN TÙY CHỈNH
# ví dụ:
# delta: 0.5-4
# theta: 4-8
# alpha: 8-13
# beta : 13-30
# gamma: 30-45
# ----------------------------
CUSTOM_BAND_NAME = "alpha_beta"
CUSTOM_BAND_LOW = 8
CUSTOM_BAND_HIGH = 30

n_freqs_in_band = 16

add_separator = True
separator_size = 2

RESAMPLE = Image.Resampling.BILINEAR if hasattr(Image, "Resampling") else Image.BILINEAR

# =========================================================
# 4) HELPER
# =========================================================
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    img_min = img.min()
    img_max = img.max()

    if img_max - img_min < 1e-8:
        return np.zeros_like(img, dtype=np.uint8)

    img = (img - img_min) / (img_max - img_min)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    return img


def get_band_freqs(band_low: float, band_high: float, n_freqs: int) -> np.ndarray:
    if n_freqs <= 0:
        raise ValueError("n_freqs phải > 0")
    if band_low >= band_high:
        raise ValueError(f"band_low phải < band_high, nhưng nhận được {band_low} >= {band_high}")

    if n_freqs == 1:
        return np.array([(band_low + band_high) / 2.0], dtype=np.float32)

    return np.linspace(band_low, band_high, n_freqs, dtype=np.float32)


def compute_band_scalogram(
    signal_1d: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs: int,
    use_log_power: bool = True,
) -> np.ndarray:
    freqs_hz = get_band_freqs(band_low, band_high, n_freqs)
    scales = pywt.frequency2scale(wavelet_name, freqs_hz / fs).astype(np.float32)

    coeffs, _ = pywt.cwt(
        data=signal_1d,
        scales=scales,
        wavelet=wavelet_name,
        sampling_period=1.0 / fs,
        method="fft"
    )

    power = np.abs(coeffs) ** 2

    if use_log_power:
        power = np.log1p(power)

    power = power[::-1, :]  # tần số cao ở trên
    return power.astype(np.float32)


def make_gray_band_vertical_scalogram(
    window_norm: np.ndarray,
    band_low: float,
    band_high: float,
    fs: float,
    wavelet_name: str,
    n_freqs_in_band: int = 16,
    use_log_power: bool = True,
    add_separator: bool = True,
    separator_size: int = 2,
    tile_size: int = 512,
) -> np.ndarray:
    """
    window_norm: (C, T)

    Output:
      gray_img: (H, W), uint8
    """
    channel_maps = []

    for ch in range(window_norm.shape[0]):
        signal_1d = window_norm[ch]

        power = compute_band_scalogram(
            signal_1d=signal_1d,
            band_low=band_low,
            band_high=band_high,
            fs=fs,
            wavelet_name=wavelet_name,
            n_freqs=n_freqs_in_band,
            use_log_power=use_log_power,
        )

        channel_maps.append(power)

        if add_separator and ch < window_norm.shape[0] - 1:
            sep = np.zeros((separator_size, power.shape[1]), dtype=np.float32)
            channel_maps.append(sep)

    vertical_img = np.concatenate(channel_maps, axis=0)   # (H_raw, W_raw)
    vertical_img_uint8 = normalize_to_uint8(vertical_img)

    img_pil = Image.fromarray(vertical_img_uint8, mode="L")
    img_pil = img_pil.resize((tile_size, tile_size), RESAMPLE)

    gray_img = np.array(img_pil, dtype=np.uint8)   # (H, W)
    return gray_img


# =========================================================
# 5) GENERATE
# =========================================================
wavelet_data_list = []

for win in tqdm(range(N), desc=f"Processing Wavelet Gray ({CUSTOM_BAND_NAME})"):
    window_data = X[win]   # (18, 1280)

    # normalize từng channel EEG
    mean = window_data.mean(axis=1, keepdims=True)
    std = window_data.std(axis=1, keepdims=True)
    window_norm = (window_data - mean) / (std + 1e-8)

    gray_img = make_gray_band_vertical_scalogram(
        window_norm=window_norm,
        band_low=CUSTOM_BAND_LOW,
        band_high=CUSTOM_BAND_HIGH,
        fs=fs,
        wavelet_name=wavelet_name,
        n_freqs_in_band=n_freqs_in_band,
        use_log_power=use_log_power,
        add_separator=add_separator,
        separator_size=separator_size,
        tile_size=tile_size,
    )  # (H, W)

    if SAVE_AS_IMAGES:
        save_path = img_out_path / f"image_{win:05d}.jpg"
        Image.fromarray(gray_img, mode="L").save(save_path, format="JPEG", quality=95)

    if SAVE_AS_NPZ:
        if SAVE_CHW:
            gray_tensor = gray_img[None, :, :]      # (1, H, W)
        else:
            gray_tensor = gray_img[:, :, None]      # (H, W, 1)

        wavelet_data_list.append(gray_tensor)

# =========================================================
# 6) SAVE NPZ
# =========================================================
if SAVE_AS_NPZ:
    X_wavelet = np.stack(wavelet_data_list, axis=0).astype(np.uint8)

    save_dict = {
        "X": X_wavelet
    }

    if y is not None:
        save_dict["y"] = y

    # copy thêm metadata cũ nếu có
    for key in ["sfreq", "window_start_sec", "window_end_sec", "source_edf", "patient_id"]:
        if key in data:
            save_dict[key] = data[key]

    np.savez_compressed(npz_out_path, **save_dict)

    print(f"Saved wavelet npz to: {npz_out_path}")
    print(f"Saved X shape: {X_wavelet.shape}")
    if y is not None:
        print(f"Saved y shape: {y.shape}")

if SAVE_AS_IMAGES:
    print(f"Saved {N} grayscale images to: {img_out_path}")

print("Done.")

X shape: (896, 18, 2048)
y shape: (896,)


Processing Wavelet Gray (alpha_beta): 100%|██████████| 896/896 [02:09<00:00,  6.90it/s]


Saved 896 grayscale images to: C:\Users\Admin\Documents\Thesis\LOSO_CV\chb13\fold_test_sid_6\test_wavelet_gray_custom_band
Done.
